Train Brand Models — Live Production Data
Trains weekly (LightGBM) and monthly (Prophet) models per brand, using hybrid model-vs-baseline selection.

**Input**: gold/erp/battery/phase2_brand_weekly_live.parquet, phase2_brand_monthly_live.parquet
**Output**: saved forecast per brand, weekly and monthly

In [0]:
%run ../../_local_config

In [0]:
%pip install lightgbm prophet

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
import json
import lightgbm as lgb
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

def wape(y_true, y_pred):
    return abs(y_true - y_pred).sum() / abs(y_true).sum()

Weekly: per-brand

In [0]:
gold_brand_weekly = read_gold(blob_service, "live/battery/data/phase2_brand_weekly_live.parquet")
gold_brand_weekly["week_start"] = pd.to_datetime(gold_brand_weekly["week_start"])
gold_brand_weekly["brand_code"] = gold_brand_weekly["brand_code"].astype("category")

print(gold_brand_weekly.shape)
print(gold_brand_weekly["brand_code"].unique())

Weekly: train/test split

In [0]:
feature_cols_brand = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w", "brand_code"]
target_col = "total_units_sold"

model_data_brand = gold_brand_weekly.dropna(subset=["lag_4w", "rolling_avg_4w", target_col]).copy()
model_data_brand = model_data_brand.sort_values("week_start")

split_idx = int(len(model_data_brand) * 0.8)
train_brand = model_data_brand.iloc[:split_idx]
test_brand = model_data_brand.iloc[split_idx:]

X_train_b, y_train_b = train_brand[feature_cols_brand], train_brand[target_col]
X_test_b, y_test_b = test_brand[feature_cols_brand], test_brand[target_col]

print(f"Train: {len(train_brand)}, Test: {len(test_brand)}")

Weekly: train, evaluate, decide model vs baseline per brand

In [0]:
model_brand = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
model_brand.fit(X_train_b, y_train_b, categorical_feature=["brand_code"])

preds_b = model_brand.predict(X_test_b)
test_brand_results = test_brand.copy()
test_brand_results["prediction"] = preds_b

brand_weekly_method = {}
for brand in test_brand_results["brand_code"].unique():
    subset = test_brand_results[test_brand_results["brand_code"] == brand]
    model_wape = wape(subset["total_units_sold"], subset["prediction"])
    baseline_wape = wape(subset["total_units_sold"], subset["rolling_avg_4w"])
    brand_weekly_method[brand] = "model" if model_wape < baseline_wape else "baseline"
    print(f"{brand} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {brand_weekly_method[brand]}")

Weekly: retrain on all data, predict next week per brand

In [0]:
final_brand_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
X_all_b = model_data_brand[feature_cols_brand]
y_all_b = model_data_brand[target_col]
final_brand_model.fit(X_all_b, y_all_b, categorical_feature=["brand_code"])

last_week_start = gold_brand_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

brand_weekly_predictions = {}
for brand in brand_weekly_method:
    brand_hist = gold_brand_weekly[gold_brand_weekly["brand_code"] == brand]

    if brand_weekly_method[brand] == "model":
        lag_val = brand_hist[brand_hist["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
        lag_val = lag_val[0] if len(lag_val) > 0 else None
        row = pd.DataFrame([{
            "week_of_year": next_week_start.isocalendar()[1],
            "month": next_week_start.month,
            "contains_month_end": int(next_week_start.month != next_week_end.month),
            "lag_4w": lag_val,
            "rolling_avg_4w": brand_hist["total_units_sold"].tail(4).mean(),
            "brand_code": brand,
        }])
        row["brand_code"] = row["brand_code"].astype("category")
        pred = final_brand_model.predict(row[feature_cols_brand])[0]
    else:
        pred = brand_hist["total_units_sold"].tail(4).mean()

    brand_weekly_predictions[brand] = max(pred, 0)
    print(f"{brand}: {brand_weekly_predictions[brand]:.0f} (week of {next_week_start.date()})")

Load monthly gold

In [0]:
gold_brand_monthly = read_gold(blob_service, "live/battery/data/phase2_brand_monthly_live.parquet")
gold_brand_monthly["month_start"] = pd.to_datetime(gold_brand_monthly["month_start"])

print(gold_brand_monthly.shape)

Monthly: per-brand Prophet, hybrid selection, 3-month forecast

In [0]:
brand_list = gold_brand_weekly["brand_code"].cat.categories

brand_monthly_method = {}
brand_monthly_forecast = {}

for brand in brand_list:
    brand_df = gold_brand_monthly[gold_brand_monthly["brand_code"] == brand][["month_start", "total_units_sold"]]
    brand_df = brand_df.rename(columns={"month_start": "ds", "total_units_sold": "y"})

    if len(brand_df) < 15 or brand_df["y"].tail(12).sum() == 0:
        method = "baseline"
    else:
        train_b = brand_df.iloc[:-3]
        test_b = brand_df.iloc[-3:]
        try:
            m_test = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
            m_test.fit(train_b)
            future_test = m_test.make_future_dataframe(periods=3, freq="MS")
            forecast_test = m_test.predict(future_test)
            test_preds = forecast_test.tail(3)["yhat"].clip(lower=0).values

            model_wape = wape(test_b["y"].values, test_preds)
            naive_pred = train_b["y"].tail(3).mean()
            baseline_wape = wape(test_b["y"].values, [naive_pred] * 3)
            method = "model" if model_wape < baseline_wape else "baseline"
            print(f"{brand} — Model: {model_wape:.3%}   Baseline: {baseline_wape:.3%}   → {method}")
        except Exception as e:
            method = "baseline"
            print(f"{brand} — Prophet failed ({e}), using baseline")

    brand_monthly_method[brand] = method

    # Final 3-month forecast, retrained on all data, using the winning method
    if method == "model":
        m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
        m_final.fit(brand_df)
        future_final = m_final.make_future_dataframe(periods=3, freq="MS")
        forecast_final = m_final.predict(future_final)
        preds = forecast_final.tail(3)["yhat"].clip(lower=0).values
        month_labels = forecast_final.tail(3)["ds"].dt.strftime("%Y-%m-%d").tolist()
    else:
        flat_value = max(brand_df["y"].tail(3).mean(), 0)
        preds = [flat_value] * 3
        last_month = brand_df["ds"].max()
        month_labels = [(last_month + pd.DateOffset(months=i)).strftime("%Y-%m-01") for i in range(1, 4)]

    brand_monthly_forecast[brand] = {"months": month_labels, "values": [float(p) for p in preds]}

for brand, data in brand_monthly_forecast.items():
    print(f"\n{brand}:")
    for month, val in zip(data["months"], data["values"]):
        print(f"  {month}: {val:.0f}")

Consistency check against overall forecast

In [0]:
blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/weekly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_weekly_overall = json.loads(stream)

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/monthly_forecast_active.json")
stream = blob_client.download_blob().readall()
active_monthly_overall = json.loads(stream)

print("Weekly consistency check:")
weekly_sum = sum(brand_weekly_predictions.values())
overall_weekly_val = next(
    (w["predicted_units"] for w in active_weekly_overall if w["week_start"] == next_week_start.strftime("%Y-%m-%d")),
    "N/A"
)
print(f"Overall: {overall_weekly_val}   Brand sum: {weekly_sum:.0f}")

print("\nMonthly consistency check:")
overall_monthly_by_date = {m["month_start"]: m["predicted_units"] for m in active_monthly_overall}
for i in range(3):
    month_label = brand_monthly_forecast[brand_list[0]]["months"][i]
    brand_sum = sum(data["values"][i] for data in brand_monthly_forecast.values())
    overall_val = overall_monthly_by_date.get(month_label, "N/A")
    print(f"{month_label} — Overall: {overall_val}   Brand sum: {brand_sum:.0f}")

Save the brand forecast

In [0]:
from src.io.storage import get_blob_service, read_gold, append_json_history, save_history_as_excel

In [0]:
import datetime

today_str = datetime.date.today().isoformat()
today = pd.Timestamp(datetime.date.today())

# Weekly: append to history, per brand
weekly_records = [
    {
        "generated_date": today_str,
        "week_start": next_week_start.strftime("%Y-%m-%d"),
        "week_end": (next_week_start + pd.Timedelta(days=6)).strftime("%Y-%m-%d"),
        "brand_code": brand,
        "predicted_units": round(float(pred))
    }
    for brand, pred in brand_weekly_predictions.items()
]

weekly_history = append_json_history(blob_service, weekly_records, "live/battery/forecasts/history/brand_weekly_forecast_history.json")

# Monthly: append to history, per brand
monthly_records = []
for brand, data in brand_monthly_forecast.items():
    for month_label, val in zip(data["months"], data["values"]):
        monthly_records.append({
            "generated_date": today_str,
            "month_start": month_label,
            "brand_code": brand,
            "predicted_units": round(float(val))
        })

monthly_history = append_json_history(blob_service, monthly_records, "live/battery/forecasts/history/brand_monthly_forecast_history.json")

print(f"Brand weekly history: {len(weekly_history)} total records")
print(f"Brand monthly history: {len(monthly_history)} total records")

In [0]:
weekly_df = pd.DataFrame(weekly_history)
weekly_df["week_end"] = pd.to_datetime(weekly_df["week_end"])
weekly_df["generated_date"] = pd.to_datetime(weekly_df["generated_date"])

active_weekly = weekly_df[weekly_df["week_end"] >= today]
active_weekly = active_weekly.sort_values("generated_date").drop_duplicates(subset=["week_start", "brand_code"], keep="last")
active_weekly = active_weekly.sort_values(["week_start", "brand_code"])

active_weekly_records = active_weekly.to_dict(orient="records")
for r in active_weekly_records:
    r["week_end"] = r["week_end"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/brand_weekly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_weekly_records, indent=2), overwrite=True)
print(f"Active brand weekly forecasts: {len(active_weekly_records)}")

In [0]:
# Monthly: active forecasts, deduped per brand+period
monthly_df = pd.DataFrame(monthly_history)
monthly_df["month_start"] = pd.to_datetime(monthly_df["month_start"])
monthly_df["generated_date"] = pd.to_datetime(monthly_df["generated_date"])
monthly_df["month_end"] = monthly_df["month_start"] + pd.offsets.MonthEnd(0)

active_monthly = monthly_df[monthly_df["month_end"] >= today]
active_monthly = active_monthly.sort_values("generated_date").drop_duplicates(subset=["month_start", "brand_code"], keep="last")
active_monthly = active_monthly.sort_values(["month_start", "brand_code"])

active_monthly_records = active_monthly.drop(columns=["month_end"]).to_dict(orient="records")
for r in active_monthly_records:
    r["month_start"] = r["month_start"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/brand_monthly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_monthly_records, indent=2), overwrite=True)
print(f"Active brand monthly forecasts: {len(active_monthly_records)}")

In [0]:
count = save_history_as_excel(blob_service, weekly_history, "live/battery/forecasts/history/brand_weekly_forecast_history.xlsx")
print(f"Saved brand weekly Excel: {count} rows")

count = save_history_as_excel(blob_service, monthly_history, "live/battery/forecasts/history/brand_monthly_forecast_history.xlsx")
print(f"Saved brand monthly Excel: {count} rows")